# Bias in Bios CLAQ Experiment

Train a CLAQ-style neural questioner on the original 28-way Bias-in-Bios profession task.

The experiment keeps the target as the original profession label. A learned actor starts from a partial concept-answer history, chooses one next concept question, receives a hard Concept-QA answer, and a neural classifier predicts the profession from the updated concept state.

This mirrors the CelebA setup more closely than the earlier heuristic prototype:

- the actor is a neural network
- the classifier is a neural network
- partial histories are sampled during training
- the actor and classifier are trained end-to-end with cross-entropy on the profession target
- evaluation rolls the actor forward for multiple question budgets

The goal is to check whether the learned questioner can reach useful confidence with a small number of concept questions.


In [ ]:
import copy
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from sentence_transformers import SentenceTransformer
from sklearn.metrics import accuracy_score, f1_score
from torch.utils.data import DataLoader, TensorDataset
from tqdm.auto import tqdm


def find_repo_root(start_path):
    start_path = Path(start_path).resolve()
    for candidate in [start_path, *start_path.parents]:
        if (candidate / "assets" / "concepts" / "bias_in_bios.csv").exists():
            return candidate
    raise FileNotFoundError("Could not find repo root from current working directory")


repo_root = find_repo_root(Path.cwd())
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from claq.models import ConceptAnswererMLP, Network
from claq.training import GradientReversal, HistorySamplingConfig, sample_history_mask, seed_everything

seed_everything(0)

sample_dir = repo_root / "artifacts" / "data" / "bias_in_bios_sample"
concepts_path = repo_root / "assets" / "concepts" / "bias_in_bios.csv"
labels_wide_path = sample_dir / "labels_wide_test_train_validation.csv"
qa_checkpoint_path = repo_root / "artifacts" / "models" / "concept_qa_bias_in_bios_concept_answerer_mlp_openai_labels.pt"
models_dir = repo_root / "artifacts" / "models"
runs_dir = repo_root / "artifacts" / "runs"
models_dir.mkdir(parents=True, exist_ok=True)
runs_dir.mkdir(parents=True, exist_ok=True)

SPLITS = ["train", "validation", "test"]
TEXT_COLUMN = "hard_text"
MAX_QUESTIONS = 20
NUM_EPOCHS = 50
BATCH_SIZE = 256
LEARNING_RATE = 1e-3
ACTOR_EPS_START = 1.0
ACTOR_EPS_END = 0.2
TRAIN_MIN_HISTORY = 0
TRAIN_MAX_HISTORY = 20
CONFIDENCE_THRESHOLDS = [0.50, 0.60, 0.70, 0.80, 0.90]

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

{
    "repo_root": str(repo_root),
    "sample_dir": str(sample_dir),
    "labels_wide_path": str(labels_wide_path),
    "qa_checkpoint_path": str(qa_checkpoint_path),
    "device": str(device),
}


In [ ]:
concepts_df = pd.read_csv(concepts_path)
concept_names = concepts_df["concept"].tolist()
labels_wide = pd.read_csv(labels_wide_path)

split_frames = {}
for split in SPLITS:
    samples = pd.read_csv(sample_dir / f"{split}.csv").reset_index(names="sample_row")
    labels = labels_wide.loc[labels_wide["split"].eq(split)].copy()
    merged = samples.merge(
        labels[["sample_row", "source_index", *concept_names]],
        on=["sample_row", "source_index"],
        how="inner",
        validate="one_to_one",
    )
    if len(merged) != len(samples):
        raise ValueError(f"Merged {len(merged)} rows for {split}, expected {len(samples)}")
    split_frames[split] = merged

profession_id_to_name = (
    split_frames["train"][["profession", "profession_name"]]
    .drop_duplicates()
    .sort_values("profession")
    .set_index("profession")["profession_name"]
    .to_dict()
)
num_professions = len(profession_id_to_name)

oracle_labels_01 = {
    split: frame[concept_names].to_numpy(dtype=np.float32)
    for split, frame in split_frames.items()
}
oracle_answers = {
    split: np.where(labels > 0, 1.0, -1.0).astype(np.float32)
    for split, labels in oracle_labels_01.items()
}
y_by_split = {
    split: frame["profession"].to_numpy(dtype=int)
    for split, frame in split_frames.items()
}
s_by_split = {
    split: frame["gender"].to_numpy(dtype=np.float32)
    for split, frame in split_frames.items()
}

{
    "split_rows": {split: len(frame) for split, frame in split_frames.items()},
    "num_professions": num_professions,
    "num_concepts": len(concept_names),
    "sensitive_target": "dataset gender label, 1=female and 0=male",
    "female_share_by_split": {split: float(values.mean()) for split, values in s_by_split.items()},
    "max_questions": MAX_QUESTIONS,
}


In [ ]:
checkpoint = torch.load(qa_checkpoint_path, map_location=device, weights_only=False)
encoder_name = checkpoint["encoder_name"]
model_config = checkpoint["model_config"]
model_config["hidden_dims"] = tuple(model_config["hidden_dims"])
concept_embeddings = checkpoint["concept_embeddings"].to(device).float()
decision_threshold = float(checkpoint.get("decision_threshold", 0.5))

concept_answerer = ConceptAnswererMLP(**model_config).to(device)
concept_answerer.load_state_dict(checkpoint["model_state_dict"])
concept_answerer.eval()

encoder = SentenceTransformer(encoder_name, device=str(device))

text_embeddings = {}
for split, frame in split_frames.items():
    text_embeddings[split] = encoder.encode(
        frame[TEXT_COLUMN].fillna("").tolist(),
        batch_size=128,
        show_progress_bar=True,
        convert_to_numpy=True,
    ).astype(np.float32)

{
    "encoder_name": encoder_name,
    "model_config": model_config,
    "decision_threshold": decision_threshold,
    "concept_embeddings": tuple(concept_embeddings.shape),
}


In [ ]:
def build_text_concept_inputs(text_batch, concept_embeddings):
    repeated_text = text_batch.repeat_interleave(len(concept_names), dim=0)
    repeated_concepts = concept_embeddings.repeat(text_batch.size(0), 1)
    return torch.cat([repeated_text, repeated_concepts], dim=1)


@torch.no_grad()
def predict_concept_scores(text_embedding_array, batch_size=256):
    concept_answerer.eval()
    scores = []
    text_tensor = torch.tensor(text_embedding_array, dtype=torch.float32, device=device)

    for start in tqdm(range(0, len(text_tensor), batch_size), desc="Predict Concept-QA scores"):
        text_batch = text_tensor[start : start + batch_size]
        inputs = build_text_concept_inputs(text_batch, concept_embeddings)
        logits = concept_answerer(inputs).view(text_batch.size(0), len(concept_names))
        scores.append(torch.sigmoid(logits).cpu().numpy())

    return np.concatenate(scores, axis=0).astype(np.float32)


qa_scores = {split: predict_concept_scores(text_embeddings[split]) for split in SPLITS}
qa_labels_01 = {
    split: (scores >= decision_threshold).astype(np.float32)
    for split, scores in qa_scores.items()
}
qa_answers = {
    split: np.where(labels > 0, 1.0, -1.0).astype(np.float32)
    for split, labels in qa_labels_01.items()
}

{
    "decision_threshold": decision_threshold,
    "qa_positive_rate": {split: float(labels.mean()) for split, labels in qa_labels_01.items()},
    "oracle_positive_rate": {split: float(labels.mean()) for split, labels in oracle_labels_01.items()},
}


In [ ]:
def make_answer_dataset(answer_matrix, labels, sensitive_labels):
    return TensorDataset(
        torch.tensor(answer_matrix, dtype=torch.float32),
        torch.tensor(labels, dtype=torch.long),
        torch.tensor(sensitive_labels, dtype=torch.float32),
    )


train_loader = DataLoader(
    make_answer_dataset(qa_answers["train"], y_by_split["train"], s_by_split["train"]),
    batch_size=BATCH_SIZE,
    shuffle=True,
)
validation_loader = DataLoader(
    make_answer_dataset(qa_answers["validation"], y_by_split["validation"], s_by_split["validation"]),
    batch_size=BATCH_SIZE,
    shuffle=False,
)
test_loader = DataLoader(
    make_answer_dataset(qa_answers["test"], y_by_split["test"], s_by_split["test"]),
    batch_size=BATCH_SIZE,
    shuffle=False,
)

sensitive_indices = torch.tensor(
    concepts_df.index[concepts_df["kind"].ne("utility")].tolist(),
    dtype=torch.long,
    device=device,
)
history_config = HistorySamplingConfig(
    min_history=TRAIN_MIN_HISTORY,
    max_history=TRAIN_MAX_HISTORY,
    non_sensitive_only=False,
)


def build_neural_claq_models(actor_eps=ACTOR_EPS_START):
    actor = Network(query_size=len(concept_names), output_size=len(concept_names), eps=actor_eps).to(device)
    classifier = Network(query_size=len(concept_names), output_size=num_professions, eps=None).to(device)
    sensitive_head = Network(query_size=len(concept_names), output_size=1, eps=None).to(device)
    return actor, classifier, sensitive_head

{
    "train_batches": len(train_loader),
    "validation_batches": len(validation_loader),
    "test_batches": len(test_loader),
    "history_config": history_config,
    "sensitive_target": "dataset gender label, 1=female and 0=male",
    "female_share_by_split": {split: float(values.mean()) for split, values in s_by_split.items()},
    "sensitive_concepts_for_query_penalty": concepts_df.loc[sensitive_indices.cpu().numpy(), "concept"].tolist(),
}


In [ ]:
def actor_eps_for_epoch(epoch, num_epochs=NUM_EPOCHS):
    if num_epochs <= 1:
        return ACTOR_EPS_END
    progress = (epoch - 1) / (num_epochs - 1)
    return float(ACTOR_EPS_START + (ACTOR_EPS_END - ACTOR_EPS_START) * progress)


def batch_metrics_from_logits(logits, labels):
    predictions = logits.argmax(dim=1)
    correct = int((predictions == labels).sum().item())
    return correct, int(labels.numel())


def run_neural_claq_epoch(
    loader,
    actor,
    classifier,
    sensitive_head,
    optimizer=None,
    lambda_s=0.0,
    lambda_c=0.0,
    train=True,
):
    actor.train(train)
    classifier.train(train)
    sensitive_head.train(train)

    total = 0
    correct = 0
    loss_total = 0.0
    task_total = 0.0
    sens_total = 0.0
    qpen_total = 0.0
    q_entropy_total = 0.0
    sensitive_query_total = 0.0
    num_batches = 0

    for answers, labels, sensitive_target in tqdm(loader, desc="Train" if train else "Evaluate", leave=False):
        answers = answers.to(device)
        labels = labels.to(device)
        sensitive_target = sensitive_target.to(device)

        with torch.set_grad_enabled(train):
            mask, masked_answers = sample_history_mask(
                answers=answers,
                config=history_config,
                sensitive_indices=sensitive_indices,
            )
            query_distribution = actor(masked_answers, mask)
            updated_answers = masked_answers + query_distribution * answers

            logits = classifier(updated_answers)
            loss_task = F.cross_entropy(logits, labels)

            sensitive_logits = sensitive_head(GradientReversal.apply(updated_answers, lambda_s)).squeeze(-1)
            loss_sensitive = F.binary_cross_entropy_with_logits(sensitive_logits, sensitive_target)

            if sensitive_indices.numel() > 0:
                sensitive_query_rate = query_distribution[:, sensitive_indices].sum(dim=1).mean()
            else:
                sensitive_query_rate = torch.zeros((), device=device)
            loss_query_penalty = lambda_c * sensitive_query_rate

            loss = loss_task + loss_sensitive + loss_query_penalty

            if train:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

        batch_correct, batch_total = batch_metrics_from_logits(logits.detach(), labels)
        correct += batch_correct
        total += batch_total
        loss_total += float(loss.detach().item())
        task_total += float(loss_task.detach().item())
        sens_total += float(loss_sensitive.detach().item())
        qpen_total += float(loss_query_penalty.detach().item())
        sensitive_query_total += float(sensitive_query_rate.detach().item())
        q_safe = query_distribution.detach().clamp_min(1e-8)
        q_entropy_total += float((-(q_safe * torch.log(q_safe)).sum(dim=1)).mean().item())
        num_batches += 1

    return {
        "accuracy": correct / max(total, 1),
        "loss": loss_total / max(num_batches, 1),
        "task_loss": task_total / max(num_batches, 1),
        "sensitive_loss": sens_total / max(num_batches, 1),
        "query_penalty": qpen_total / max(num_batches, 1),
        "sensitive_query_rate": sensitive_query_total / max(num_batches, 1),
        "query_entropy": q_entropy_total / max(num_batches, 1),
    }


def fit_neural_claq(
    run_name,
    lambda_s=0.0,
    lambda_c=0.0,
    num_epochs=NUM_EPOCHS,
    seed=0,
):
    seed_everything(seed)
    actor, classifier, sensitive_head = build_neural_claq_models(actor_eps=ACTOR_EPS_START)
    optimizer = torch.optim.Adam(
        list(actor.parameters()) + list(classifier.parameters()) + list(sensitive_head.parameters()),
        lr=LEARNING_RATE,
    )

    history = []
    best = {"validation_accuracy": -1.0}
    for epoch in tqdm(range(1, num_epochs + 1), desc=f"{run_name} epochs"):
        actor.change_eps(actor_eps_for_epoch(epoch, num_epochs=num_epochs))
        train_metrics = run_neural_claq_epoch(
            train_loader,
            actor,
            classifier,
            sensitive_head,
            optimizer=optimizer,
            lambda_s=lambda_s,
            lambda_c=lambda_c,
            train=True,
        )
        validation_metrics = run_neural_claq_epoch(
            validation_loader,
            actor,
            classifier,
            sensitive_head,
            lambda_s=lambda_s,
            lambda_c=lambda_c,
            train=False,
        )
        row = {
            "run_name": run_name,
            "epoch": epoch,
            "lambda_s": lambda_s,
            "lambda_c": lambda_c,
            "actor_eps": actor.eps,
            **{f"train_{key}": value for key, value in train_metrics.items()},
            **{f"validation_{key}": value for key, value in validation_metrics.items()},
        }
        history.append(row)

        if validation_metrics["accuracy"] >= best["validation_accuracy"]:
            best = {
                "validation_accuracy": validation_metrics["accuracy"],
                "epoch": epoch,
                "actor_state_dict": copy.deepcopy(actor.state_dict()),
                "classifier_state_dict": copy.deepcopy(classifier.state_dict()),
                "sensitive_head_state_dict": copy.deepcopy(sensitive_head.state_dict()),
                "history_row": row,
            }

    actor.load_state_dict(best["actor_state_dict"])
    classifier.load_state_dict(best["classifier_state_dict"])
    sensitive_head.load_state_dict(best["sensitive_head_state_dict"])
    return {
        "run_name": run_name,
        "lambda_s": lambda_s,
        "lambda_c": lambda_c,
        "history": pd.DataFrame(history),
        "best": best,
        "actor": actor,
        "classifier": classifier,
        "sensitive_head": sensitive_head,
    }


In [ ]:
@torch.no_grad()
def rollout_neural_claq(actor, classifier, loader, run_name, max_questions=MAX_QUESTIONS):
    actor.eval()
    classifier.eval()
    curve_rows = []
    detail_rows = []
    query_rows = []
    row_offset = 0

    for answers, labels, _sensitive_target in tqdm(loader, desc=f"Rollout {run_name}"):
        answers = answers.to(device)
        labels = labels.to(device)
        batch_size = answers.size(0)
        mask = torch.zeros_like(answers)
        masked_answers = torch.zeros_like(answers)
        row_indices = np.arange(row_offset, row_offset + batch_size)

        for budget in range(max_questions + 1):
            logits = classifier(masked_answers)
            probabilities = F.softmax(logits, dim=1)
            confidence, predictions = probabilities.max(dim=1)

            detail_rows.append(
                pd.DataFrame(
                    {
                        "run_name": run_name,
                        "budget": budget,
                        "row_idx": row_indices,
                        "label": labels.cpu().numpy(),
                        "prediction": predictions.cpu().numpy(),
                        "confidence": confidence.cpu().numpy(),
                        "correct": (predictions == labels).cpu().numpy(),
                    }
                )
            )

            if budget == max_questions:
                break

            query_distribution = actor(masked_answers, mask)
            chosen_queries = query_distribution.argmax(dim=1)
            query_rows.append(
                pd.DataFrame(
                    {
                        "run_name": run_name,
                        "budget": budget + 1,
                        "row_idx": row_indices,
                        "query_index": chosen_queries.cpu().numpy(),
                        "query_concept": [concept_names[idx] for idx in chosen_queries.cpu().numpy()],
                    }
                )
            )
            masked_answers = masked_answers + query_distribution * answers
            mask = torch.clamp(mask + query_distribution, 0.0, 1.0)

        row_offset += batch_size

    detail_df = pd.concat(detail_rows, ignore_index=True)
    query_df = pd.concat(query_rows, ignore_index=True)

    for budget, budget_detail in detail_df.groupby("budget"):
        curve_rows.append(
            {
                "run_name": run_name,
                "budget": int(budget),
                "accuracy": accuracy_score(budget_detail["label"], budget_detail["prediction"]),
                "macro_f1": f1_score(budget_detail["label"], budget_detail["prediction"], average="macro"),
                "mean_confidence": float(budget_detail["confidence"].mean()),
            }
        )

    return pd.DataFrame(curve_rows).sort_values("budget"), detail_df, query_df


def summarize_confidence_stopping(detail_df, thresholds=CONFIDENCE_THRESHOLDS):
    rows = []
    run_name = detail_df["run_name"].iloc[0]
    for threshold in thresholds:
        reached = detail_df.loc[detail_df["confidence"].ge(threshold)].copy()
        if reached.empty:
            rows.append(
                {
                    "run_name": run_name,
                    "confidence_threshold": threshold,
                    "coverage": 0.0,
                    "mean_questions_when_reached": np.nan,
                    "accuracy_when_reached": np.nan,
                }
            )
            continue

        first_reached = reached.sort_values("budget").drop_duplicates("row_idx", keep="first")
        rows.append(
            {
                "run_name": run_name,
                "confidence_threshold": threshold,
                "coverage": len(first_reached) / detail_df["row_idx"].nunique(),
                "mean_questions_when_reached": first_reached["budget"].mean(),
                "accuracy_when_reached": first_reached["correct"].mean(),
            }
        )
    return pd.DataFrame(rows)


def summarize_query_usage(query_df):
    return (
        query_df.groupby(["run_name", "budget", "query_concept"])
        .size()
        .reset_index(name="count")
        .sort_values(["run_name", "budget", "count"], ascending=[True, True, False])
    )


In [ ]:
baseline_run = fit_neural_claq(
    run_name="neural_baseline_lambda_s_0",
    lambda_s=0.0,
    lambda_c=0.0,
    seed=0,
)

claq_run = fit_neural_claq(
    run_name="neural_claq_lambda_s_0_4",
    lambda_s=0.4,
    lambda_c=0.0,
    seed=0,
)

trained_runs = [baseline_run, claq_run]

history_df = pd.concat([run["history"] for run in trained_runs], ignore_index=True)
history_df.groupby("run_name").tail(1)


In [ ]:
curve_frames = []
detail_frames = []
query_frames = []

for run in trained_runs:
    curve, detail, queries = rollout_neural_claq(
        actor=run["actor"],
        classifier=run["classifier"],
        loader=test_loader,
        run_name=run["run_name"],
    )
    curve_frames.append(curve)
    detail_frames.append(detail)
    query_frames.append(queries)

questioner_curve = pd.concat(curve_frames, ignore_index=True)
questioner_detail = pd.concat(detail_frames, ignore_index=True)
questioner_queries = pd.concat(query_frames, ignore_index=True)
stopping_summary = pd.concat(
    [summarize_confidence_stopping(detail) for detail in detail_frames],
    ignore_index=True,
)
query_usage = summarize_query_usage(questioner_queries)

questioner_curve.pivot(index="budget", columns="run_name", values="accuracy").head(MAX_QUESTIONS + 1)


In [ ]:
def format_query_path(query_df, run_name, row_idx, max_budget=10):
    path = query_df.loc[
        query_df["run_name"].eq(run_name)
        & query_df["row_idx"].eq(row_idx)
        & query_df["budget"].le(max_budget)
    ].sort_values("budget")
    return " -> ".join(path["query_concept"].tolist())


def prediction_snapshot(detail_df, run_name, row_idx, budget):
    row = detail_df.loc[
        detail_df["run_name"].eq(run_name)
        & detail_df["row_idx"].eq(row_idx)
        & detail_df["budget"].eq(budget)
    ].iloc[0]
    return {
        "prediction": profession_id_to_name[int(row["prediction"])],
        "confidence": round(float(row["confidence"]), 3),
        "correct": bool(row["correct"]),
    }


def choose_comparison_rows(detail_df, max_rows=6):
    final_budget = int(detail_df["budget"].max())
    final_predictions = detail_df.loc[detail_df["budget"].eq(final_budget)].pivot(
        index="row_idx",
        columns="run_name",
        values="prediction",
    )
    final_correct = detail_df.loc[detail_df["budget"].eq(final_budget)].pivot(
        index="row_idx",
        columns="run_name",
        values="correct",
    )
    differing_rows = final_predictions.nunique(axis=1).gt(1) | final_correct.nunique(axis=1).gt(1)
    selected = final_predictions.index[differing_rows].tolist()
    if len(selected) < max_rows:
        best_budget = (
            detail_df.sort_values(["row_idx", "budget"])
            .drop_duplicates(["run_name", "row_idx"], keep="last")
            .groupby("row_idx")["confidence"]
            .std()
            .sort_values(ascending=False)
            .index.tolist()
        )
        selected.extend([row_idx for row_idx in best_budget if row_idx not in selected])
    return selected[:max_rows]


def build_sample_comparison(detail_df, query_df, budgets=(5, 10, 20), max_query_budget=10, max_rows=6):
    rows = []
    run_names = sorted(detail_df["run_name"].unique())
    comparison_rows = choose_comparison_rows(detail_df, max_rows=max_rows)

    for row_idx in comparison_rows:
        test_row = split_frames["test"].iloc[int(row_idx)]
        row = {
            "row_idx": int(row_idx),
            "sample_row": int(test_row["sample_row"]),
            "true_profession": test_row["profession_name"],
            "biography": test_row[TEXT_COLUMN][:500],
        }
        for run_name in run_names:
            row[f"{run_name}_first_{max_query_budget}_questions"] = format_query_path(
                query_df,
                run_name,
                row_idx,
                max_budget=max_query_budget,
            )
            for budget in budgets:
                snapshot = prediction_snapshot(detail_df, run_name, row_idx, budget)
                prefix = f"{run_name}_budget_{budget}"
                row[f"{prefix}_prediction"] = snapshot["prediction"]
                row[f"{prefix}_confidence"] = snapshot["confidence"]
                row[f"{prefix}_correct"] = snapshot["correct"]
        rows.append(row)

    return pd.DataFrame(rows)


sample_comparison = build_sample_comparison(questioner_detail, questioner_queries)
sample_comparison


In [ ]:
pd.set_option("display.max_colwidth", 500)

overview_cols = ["row_idx", "sample_row", "true_profession", "biography"]
display(sample_comparison[overview_cols])

question_cols = [col for col in sample_comparison.columns if "questions" in col]
display(sample_comparison[["row_idx", *question_cols]])

prediction_cols = [
    col for col in sample_comparison.columns
    if "budget" in col and ("prediction" in col or "confidence" in col or "correct" in col)
]
display(sample_comparison[["row_idx", *prediction_cols]])


In [ ]:
best_budget_rows = (
    questioner_curve.sort_values(["run_name", "accuracy"], ascending=[True, False])
    .groupby("run_name")
    .head(1)
    .reset_index(drop=True)
)

{
    "best_budget_by_run": best_budget_rows.to_dict(orient="records"),
    "confidence_stopping": stopping_summary.to_dict(orient="records"),
    "top_queries_first_5_budgets": query_usage.loc[query_usage["budget"].le(5)].groupby(["run_name", "budget"]).head(5).to_dict(orient="records"),
}


In [ ]:
experiment_name = "bias_in_bios_claq_experiment"
curve_path = runs_dir / f"{experiment_name}_curve.csv"
detail_path = runs_dir / f"{experiment_name}_detail.csv"
stopping_path = runs_dir / f"{experiment_name}_stopping.csv"
queries_path = runs_dir / f"{experiment_name}_queries.csv"
history_path = runs_dir / f"{experiment_name}_history.csv"
query_usage_path = runs_dir / f"{experiment_name}_query_usage.csv"
sample_comparison_path = runs_dir / f"{experiment_name}_sample_comparison.csv"
summary_path = runs_dir / f"{experiment_name}_summary.json"
checkpoint_path = models_dir / f"{experiment_name}.pt"

questioner_curve.to_csv(curve_path, index=False, lineterminator="\n")
questioner_detail.to_csv(detail_path, index=False, lineterminator="\n")
stopping_summary.to_csv(stopping_path, index=False, lineterminator="\n")
questioner_queries.to_csv(queries_path, index=False, lineterminator="\n")
history_df.to_csv(history_path, index=False, lineterminator="\n")
query_usage.to_csv(query_usage_path, index=False, lineterminator="\n")
sample_comparison.to_csv(sample_comparison_path, index=False, lineterminator="\n")

torch.save(
    {
        "experiment_name": experiment_name,
        "concept_names": concept_names,
        "profession_id_to_name": profession_id_to_name,
        "decision_threshold": decision_threshold,
        "sensitive_target": "dataset gender label, 1=female and 0=male",
        "max_questions": MAX_QUESTIONS,
        "num_epochs": NUM_EPOCHS,
        "batch_size": BATCH_SIZE,
        "learning_rate": LEARNING_RATE,
        "history_config": {
            "min_history": history_config.min_history,
            "max_history": history_config.max_history,
            "non_sensitive_only": history_config.non_sensitive_only,
        },
        "runs": {
            run["run_name"]: {
                "lambda_s": run["lambda_s"],
                "lambda_c": run["lambda_c"],
                "best": run["best"],
                "actor_state_dict": run["actor"].state_dict(),
                "classifier_state_dict": run["classifier"].state_dict(),
                "sensitive_head_state_dict": run["sensitive_head"].state_dict(),
            }
            for run in trained_runs
        },
    },
    checkpoint_path,
)

summary = {
    "experiment_name": experiment_name,
    "target": "original_28_way_profession",
    "answer_source": "hard Concept-QA answers converted to -1/+1",
    "training": "neural actor/classifier trained end-to-end on sampled partial histories",
    "decision_threshold": decision_threshold,
    "sensitive_target": "dataset gender label, 1=female and 0=male",
    "max_questions": MAX_QUESTIONS,
    "num_epochs": NUM_EPOCHS,
    "batch_size": BATCH_SIZE,
    "learning_rate": LEARNING_RATE,
    "history_config": {
        "min_history": history_config.min_history,
        "max_history": history_config.max_history,
        "non_sensitive_only": history_config.non_sensitive_only,
    },
    "num_concepts": len(concept_names),
    "num_professions": num_professions,
    "best_budget_by_run": best_budget_rows.to_dict(orient="records"),
    "outputs": {
        "curve": str(curve_path.relative_to(repo_root)),
        "detail": str(detail_path.relative_to(repo_root)),
        "stopping": str(stopping_path.relative_to(repo_root)),
        "queries": str(queries_path.relative_to(repo_root)),
        "history": str(history_path.relative_to(repo_root)),
        "query_usage": str(query_usage_path.relative_to(repo_root)),
        "sample_comparison": str(sample_comparison_path.relative_to(repo_root)),
        "checkpoint": str(checkpoint_path.relative_to(repo_root)),
    },
}
with summary_path.open("w", encoding="utf-8") as handle:
    json.dump(summary, handle, indent=2)

summary["outputs"] | {"summary": str(summary_path.relative_to(repo_root))}
